# Advanced Problems with Solutions: Python Integers (`int`)
This notebook contains advanced practice problems about Python's arbitrary-precision integers.

Topics covered:

- `int` as an object
- arbitrary precision and `bit_length`
- memory usage with `sys.getsizeof`
- performance effects of integer magnitude
- modular arithmetic best practices
- integer parsing safety limits
- integer identity and caching caveats
- binary serialization with `int.to_bytes` and `int.from_bytes`

> Run the notebook top to bottom with Python 3.13.


In [1]:
import math
import random
import statistics
import sys
from timeit import repeat

print(sys.version)


3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


## Problem 1 — Inspect integer object behavior

Prove that integers are objects, `bool` is a subclass of `int`, and `is` should not be used for numeric equality.


In [2]:
# Solution

values = [0, 1, -1, 10 ** 100, True, False]

for value in values:
    print(f"{value!r:>8} -> type={type(value).__name__}, isinstance(value, int)={isinstance(value, int)}")

print()
print("issubclass(bool, int):", issubclass(bool, int))
print("True + True + 10:", True + True + 10)

a = int("100000000000000000000")
b = int("100000000000000000000")

print()
print("a == b:", a == b)
print("a is b:", a is b)

# Best practice: use == for equality, not is.


       0 -> type=int, isinstance(value, int)=True
       1 -> type=int, isinstance(value, int)=True
      -1 -> type=int, isinstance(value, int)=True
10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000 -> type=int, isinstance(value, int)=True
    True -> type=bool, isinstance(value, int)=True
   False -> type=bool, isinstance(value, int)=True

issubclass(bool, int): True
True + True + 10: 12

a == b: True
a is b: False


### Explanation

`==` compares numeric value. `is` compares object identity. Some small integers may appear to share identity in CPython because of implementation-level caching, but correct numeric code must use `==`.


## Problem 2 — Profile integer memory usage

Write `int_profile(n)` returning the sign, bit length, minimum magnitude bytes, and `sys.getsizeof(n)`.


In [3]:
# Solution

def int_profile(n: int) -> dict[str, int | str]:
    if not isinstance(n, int):
        raise TypeError("n must be an int")

    bits = n.bit_length()

    if n == 0:
        decimal_digits = 1
    else:
        decimal_digits = math.floor(bits * math.log10(2)) + 1

    return {
        "value_preview": str(n) if abs(n) < 10 ** 20 else f"<large integer with about {decimal_digits} decimal digits>",
        "sign": "negative" if n < 0 else "zero" if n == 0 else "positive",
        "bit_length": bits,
        "min_unsigned_bytes_for_magnitude": (bits + 7) // 8,
        "sys_getsizeof_bytes": sys.getsizeof(n),
    }


for n in [0, 1, 2 ** 30 - 1, 2 ** 30, 2 ** 60, 2 ** 1000]:
    print(int_profile(n))


{'value_preview': '0', 'sign': 'zero', 'bit_length': 0, 'min_unsigned_bytes_for_magnitude': 0, 'sys_getsizeof_bytes': 28}
{'value_preview': '1', 'sign': 'positive', 'bit_length': 1, 'min_unsigned_bytes_for_magnitude': 1, 'sys_getsizeof_bytes': 28}
{'value_preview': '1073741823', 'sign': 'positive', 'bit_length': 30, 'min_unsigned_bytes_for_magnitude': 4, 'sys_getsizeof_bytes': 28}
{'value_preview': '1073741824', 'sign': 'positive', 'bit_length': 31, 'min_unsigned_bytes_for_magnitude': 4, 'sys_getsizeof_bytes': 32}
{'value_preview': '1152921504606846976', 'sign': 'positive', 'bit_length': 61, 'min_unsigned_bytes_for_magnitude': 8, 'sys_getsizeof_bytes': 36}
{'value_preview': '<large integer with about 302 decimal digits>', 'sign': 'positive', 'bit_length': 1001, 'min_unsigned_bytes_for_magnitude': 126, 'sys_getsizeof_bytes': 160}


### Explanation

`bit_length()` is the most reliable high-level way to reason about an integer's magnitude. Exact `sys.getsizeof` values are implementation-dependent, but larger magnitudes require more storage.


## Problem 3 — Detect memory growth steps

Scan powers of two and print only the exponents where `sys.getsizeof(2**k)` changes.


In [4]:
# Solution

def memory_steps(max_exponent: int = 256) -> list[tuple[int, int, int]]:
    if max_exponent < 0:
        raise ValueError("max_exponent must be non-negative")

    steps = []
    previous_size = None

    for k in range(max_exponent + 1):
        n = 2 ** k
        size = sys.getsizeof(n)

        if size != previous_size:
            steps.append((k, n.bit_length(), size))
            previous_size = size

    return steps


print("exponent | bit_length | sys.getsizeof")
print("-" * 42)
for exponent, bit_length, size in memory_steps(256):
    print(f"{exponent:8d} | {bit_length:10d} | {size:13d}")


exponent | bit_length | sys.getsizeof
------------------------------------------
       0 |          1 |            28
      30 |         31 |            32
      60 |         61 |            36
      90 |         91 |            40
     120 |        121 |            44
     150 |        151 |            48
     180 |        181 |            52
     210 |        211 |            56
     240 |        241 |            60


### Explanation

Integer storage grows in chunks because CPython stores large integers using internal digits, not one Python object per bit. Treat those chunk sizes as implementation details.


## Problem 4 — Minimum bytes for integer magnitude

Create `magnitude_bytes(n)` returning the minimum number of bytes needed to store `abs(n)`. For `0`, return `0`.


In [5]:
# Solution

def magnitude_bytes(n: int) -> int:
    if not isinstance(n, int):
        raise TypeError("n must be an int")
    return (abs(n).bit_length() + 7) // 8


for n in [0, 1, -1, 255, 256, -256, 2 ** 64 - 1, -(2 ** 64), 10 ** 100]:
    print(f"{n!r:>25} -> {magnitude_bytes(n)} byte(s)")


                        0 -> 0 byte(s)
                        1 -> 1 byte(s)
                       -1 -> 1 byte(s)
                      255 -> 1 byte(s)
                      256 -> 2 byte(s)
                     -256 -> 2 byte(s)
     18446744073709551615 -> 8 byte(s)
    -18446744073709551616 -> 9 byte(s)
10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000 -> 42 byte(s)


## Problem 5 — Minimal byte serialization

Write helpers that serialize and deserialize integers with `to_bytes` and `from_bytes`. Support unsigned non-negative integers and signed integers.


In [6]:
# Solution

def int_to_minimal_bytes(n: int, *, signed: bool = False, byteorder: str = "big") -> bytes:
    if not isinstance(n, int):
        raise TypeError("n must be an int")
    if byteorder not in {"big", "little"}:
        raise ValueError("byteorder must be 'big' or 'little'")
    if not signed and n < 0:
        raise ValueError("cannot serialize a negative integer with signed=False")
    if n == 0:
        return b"\x00"

    if not signed:
        length = (n.bit_length() + 7) // 8
        return n.to_bytes(length, byteorder=byteorder, signed=False)

    for length in range(1, max(2, (abs(n).bit_length() + 8) // 8 + 2)):
        try:
            data = n.to_bytes(length, byteorder=byteorder, signed=True)
        except OverflowError:
            continue
        if int.from_bytes(data, byteorder=byteorder, signed=True) == n:
            return data

    raise OverflowError("could not find a minimal signed representation")


def int_from_bytes(data: bytes | bytearray | memoryview, *, signed: bool = False, byteorder: str = "big") -> int:
    if byteorder not in {"big", "little"}:
        raise ValueError("byteorder must be 'big' or 'little'")
    if not isinstance(data, (bytes, bytearray, memoryview)):
        raise TypeError("data must be bytes-like")
    if len(data) == 0:
        raise ValueError("empty byte string is not accepted by this helper")
    return int.from_bytes(data, byteorder=byteorder, signed=signed)


for n in [0, 1, 255, 256, 2 ** 64 - 1, 10 ** 100]:
    data = int_to_minimal_bytes(n)
    assert int_from_bytes(data) == n
    print(f"unsigned {n!r:>24} -> {data.hex()}")

print()

for n in [0, 1, -1, 127, 128, -128, -129, 2 ** 64, -(2 ** 64)]:
    data = int_to_minimal_bytes(n, signed=True)
    assert int_from_bytes(data, signed=True) == n
    print(f"signed   {n!r:>24} -> {data.hex()}")


unsigned                        0 -> 00
unsigned                        1 -> 01
unsigned                      255 -> ff
unsigned                      256 -> 0100
unsigned     18446744073709551615 -> ffffffffffffffff
unsigned 10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000 -> 1249ad2594c37ceb0b2784c4ce0bf38ace408e211a7caab24308a82e8f10000000000000000000000000

signed                          0 -> 00
signed                          1 -> 01
signed                         -1 -> ff
signed                        127 -> 7f
signed                        128 -> 0080
signed                       -128 -> 80
signed                       -129 -> ff7f
signed       18446744073709551616 -> 010000000000000000
signed      -18446744073709551616 -> ff0000000000000000


### Explanation

Unsigned serialization depends only on magnitude. Signed serialization uses two's-complement representation, so positive values such as `128` may need an extra leading byte to avoid being decoded as negative.


## Problem 6 — Benchmark integer multiplication by magnitude

Use `timeit.repeat` to compare `x * 2` for integers with different bit lengths.


In [7]:
# Solution

def multiply_by_two(x: int) -> int:
    return x * 2


def benchmark_multiplication(values: list[int], *, number: int = 100_000, repeat_count: int = 7):
    rows = []

    for x in values:
        timings = repeat(
            stmt="multiply_by_two(x)",
            globals={"multiply_by_two": multiply_by_two, "x": x},
            number=number,
            repeat=repeat_count,
        )

        # Safe digit estimate (no str conversion)
        digits = 1 if x == 0 else math.floor(x.bit_length() * math.log10(2)) + 1

        rows.append((
            x.bit_length(),
            digits,
            statistics.median(timings),
            min(timings)
        ))

    return rows


values = [10, 2 ** 100, 2 ** 1_000, 2 ** 10_000, 2 ** 100_000]

print("bit_length | decimal_digits | median_seconds | best_seconds")
print("-" * 64)
for bits, digits, median_seconds, best_seconds in benchmark_multiplication(values):
    print(f"{bits:10d} | {digits:14d} | {median_seconds:14.6f} | {best_seconds:12.6f}")


bit_length | decimal_digits | median_seconds | best_seconds
----------------------------------------------------------------
         4 |              2 |       0.007009 |     0.005577
       101 |             31 |       0.009258 |     0.007583
      1001 |            302 |       0.012145 |     0.011115
     10001 |           3011 |       0.052835 |     0.050677
    100001 |          30104 |       0.410833 |     0.408155


### Explanation

Large integer arithmetic is slower because operations must process more internal digits. `timeit.repeat` reduces noise compared with a single manual timing.


## Problem 7 — Modular exponentiation

Compare `(base ** exponent) % modulus` with `pow(base, exponent, modulus)`.


In [8]:
# Solution

def slow_mod_power(base: int, exponent: int, modulus: int) -> int:
    return (base ** exponent) % modulus


def fast_mod_power(base: int, exponent: int, modulus: int) -> int:
    return pow(base, exponent, modulus)


base = 987654321987654321
exponent = 20_000
modulus = 1_000_000_007

assert slow_mod_power(base, exponent, modulus) == fast_mod_power(base, exponent, modulus)
print("answer:", fast_mod_power(base, exponent, modulus))

slow_times = repeat("slow_mod_power(base, exponent, modulus)", globals=globals(), number=1, repeat=3)
fast_times = repeat("fast_mod_power(base, exponent, modulus)", globals=globals(), number=1, repeat=7)

print("slow median:", statistics.median(slow_times))
print("fast median:", statistics.median(fast_times))
print("speedup factor:", statistics.median(slow_times) / statistics.median(fast_times))


answer: 458883092
slow median: 0.09335209988057613
fast median: 8.996576070785522e-07
speedup factor: 103764.03105590062


### Explanation

`pow(base, exponent, modulus)` performs modular exponentiation directly and avoids constructing a huge intermediate integer.


## Problem 8 — Integer parsing limits

Inspect Python's decimal string-to-int safety limit and compare decimal parsing with hexadecimal parsing.


In [9]:
# Solution

limit = sys.get_int_max_str_digits()
print("decimal digit conversion limit:", limit)

too_long_decimal = "9" * (limit + 1)

try:
    int(too_long_decimal)
except ValueError as exc:
    print("decimal conversion failed as expected:")
    print(type(exc).__name__, str(exc).splitlines()[0])

large_hex = "f" * (limit + 1)
n = int(large_hex, 16)

print()
print("hex digits parsed:", len(large_hex))
print("result bit_length:", n.bit_length())
print("result object size:", sys.getsizeof(n))


decimal digit conversion limit: 4300
decimal conversion failed as expected:
ValueError Exceeds the limit (4300 digits) for integer string conversion: value has 4301 digits; use sys.set_int_max_str_digits() to increase the limit

hex digits parsed: 4301
result bit_length: 17204
result object size: 2320


### Explanation

Decimal conversion is comparatively expensive for extremely long inputs, so Python applies a safety limit. Bases that are powers of two, such as binary, octal, and hexadecimal, map more directly to bits.


## Problem 9 — Safe integer parser

Implement `parse_int_safe(text, base=10, max_digits=None)` with validation before calling `int`.


In [10]:
# Solution

def parse_int_safe(text: str, *, base: int = 10, max_digits: int | None = None) -> int:
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    if not (2 <= base <= 36):
        raise ValueError("base must be between 2 and 36")

    stripped = text.strip()
    if not stripped:
        raise ValueError("empty string is not a valid integer")
    if "_" in stripped:
        raise ValueError("underscores are not accepted")

    signless = stripped[1:] if stripped[0] in "+-" else stripped
    if not signless:
        raise ValueError("sign without digits is not a valid integer")

    if max_digits is not None and len(signless) > max_digits:
        raise ValueError(f"too many digits: {len(signless)} > {max_digits}")

    valid_digits = "0123456789abcdefghijklmnopqrstuvwxyz"[:base]
    invalid = [ch for ch in signless.lower() if ch not in valid_digits]
    if invalid:
        raise ValueError(f"invalid digit {invalid[0]!r} for base {base}")

    return int(stripped, base)


for text, base in [("123", 10), ("   -123   ", 10), ("ff", 16), ("101010", 2), ("z", 36)]:
    print(f"{text!r}, base={base} -> {parse_int_safe(text, base=base, max_digits=10)}")

for text in ["", "+", "1_000", "102", "abcdef"]:
    try:
        parse_int_safe(text, base=2, max_digits=5)
    except ValueError as exc:
        print(f"{text!r} rejected:", exc)


'123', base=10 -> 123
'   -123   ', base=10 -> -123
'ff', base=16 -> 255
'101010', base=2 -> 42
'z', base=36 -> 35
'' rejected: empty string is not a valid integer
'+' rejected: sign without digits is not a valid integer
'1_000' rejected: underscores are not accepted
'102' rejected: invalid digit '2' for base 2
'abcdef' rejected: too many digits: 6 > 5


## Problem 10 — Population count with `bit_count`

Compare `bin(n).count("1")` with `n.bit_count()`.


In [11]:
# Solution

def bit_count_slow(n: int) -> int:
    if n < 0:
        raise ValueError("this exercise expects a non-negative integer")
    return bin(n).count("1")


def bit_count_fast(n: int) -> int:
    if n < 0:
        raise ValueError("this exercise expects a non-negative integer")
    return n.bit_count()


rng = random.Random(42)

for _ in range(1_000):
    n = rng.getrandbits(rng.randrange(0, 10_000))
    assert bit_count_slow(n) == bit_count_fast(n)

print("randomized correctness tests passed")

large = rng.getrandbits(1_000_000)

slow_times = repeat("bit_count_slow(large)", globals=globals(), number=1, repeat=3)
fast_times = repeat("bit_count_fast(large)", globals=globals(), number=10, repeat=5)

print("slow median per call:", statistics.median(slow_times))
print("fast median per call:", statistics.median(fast_times) / 10)
print("speedup factor:", statistics.median(slow_times) / (statistics.median(fast_times) / 10))


randomized correctness tests passed
slow median per call: 0.011102200485765934
fast median per call: 9.102998301386833e-05
speedup factor: 121.96201864691686


### Explanation

`bin(n).count("1")` creates a large temporary string. `n.bit_count()` works directly with the integer representation and is preferred.


## Problem 11 — Integer range classifier

Classify whether an integer fits signed or unsigned 8-bit, 32-bit, or 64-bit ranges.


In [12]:
# Solution

def classify_int(n: int) -> str:
    if not isinstance(n, int):
        raise TypeError("n must be an int")
    if n == 0:
        return "zero"
    if -(2 ** 7) <= n <= 2 ** 7 - 1:
        return "fits signed 8-bit"
    if 0 <= n <= 2 ** 8 - 1:
        return "fits unsigned 8-bit only"
    if -(2 ** 31) <= n <= 2 ** 31 - 1:
        return "fits signed 32-bit"
    if 0 <= n <= 2 ** 32 - 1:
        return "fits unsigned 32-bit only"
    if -(2 ** 63) <= n <= 2 ** 63 - 1:
        return "fits signed 64-bit"
    if 0 <= n <= 2 ** 64 - 1:
        return "fits unsigned 64-bit only"
    return "larger than 64-bit"


boundary_values = [
    0,
    -129, -128, -127,
    127, 128, 255, 256,
    -(2 ** 31) - 1, -(2 ** 31), 2 ** 31 - 1, 2 ** 31,
    2 ** 32 - 1, 2 ** 32,
    -(2 ** 63) - 1, -(2 ** 63), 2 ** 63 - 1, 2 ** 63,
    2 ** 64 - 1, 2 ** 64,
]

for n in boundary_values:
    print(f"{n:>24} -> {classify_int(n)}")


                       0 -> zero
                    -129 -> fits signed 32-bit
                    -128 -> fits signed 8-bit
                    -127 -> fits signed 8-bit
                     127 -> fits signed 8-bit
                     128 -> fits unsigned 8-bit only
                     255 -> fits unsigned 8-bit only
                     256 -> fits signed 32-bit
             -2147483649 -> fits signed 64-bit
             -2147483648 -> fits signed 32-bit
              2147483647 -> fits signed 32-bit
              2147483648 -> fits unsigned 32-bit only
              4294967295 -> fits unsigned 32-bit only
              4294967296 -> fits signed 64-bit
    -9223372036854775809 -> larger than 64-bit
    -9223372036854775808 -> fits signed 64-bit
     9223372036854775807 -> fits signed 64-bit
     9223372036854775808 -> fits unsigned 64-bit only
    18446744073709551615 -> fits unsigned 64-bit only
    18446744073709551616 -> larger than 64-bit


## Problem 12 — Robust benchmarking helper

Write `median_time_ns(func, *args, repeats=7, loops=1000)` and compare addition, multiplication, and squaring for small and large integers.


In [13]:
# Solution

def median_time_ns(func, *args, repeats: int = 7, loops: int = 1_000) -> float:
    if repeats < 1:
        raise ValueError("repeats must be positive")
    if loops < 1:
        raise ValueError("loops must be positive")

    timer_values = repeat(
        stmt="func(*args)",
        globals={"func": func, "args": args},
        number=loops,
        repeat=repeats,
    )

    return statistics.median(timer_values) * 1_000_000_000 / loops


def add(a: int, b: int) -> int:
    return a + b


def mul(a: int, b: int) -> int:
    return a * b


def square(a: int) -> int:
    return a * a


small = 12345
large = 2 ** 100_000 + 12345

benchmarks = [
    ("small add", add, small, small),
    ("large add", add, large, large),
    ("small multiply", mul, small, small),
    ("large multiply", mul, large, large),
    ("small square", square, small),
    ("large square", square, large),
]

print("operation        | median ns/call")
print("-" * 36)

for name, func, *args in benchmarks:
    loops = 100_000 if "small" in name else 100
    ns = median_time_ns(func, *args, repeats=7, loops=loops)
    print(f"{name:<16} | {ns:14.1f}")


operation        | median ns/call
------------------------------------
small add        |          123.8
large add        |         4237.0
small multiply   |          136.2
large multiply   |       388634.0
small square     |          158.8
large square     |       442000.0


## Problem 13 — Avoid huge intermediates

Write `last_decimal_digits(a, b, k)` to compute only the last `k` decimal digits of `a ** b`.


In [14]:
# Solution

def last_decimal_digits(a: int, b: int, k: int) -> int:
    if not all(isinstance(x, int) for x in (a, b, k)):
        raise TypeError("a, b, and k must be integers")
    if b < 0:
        raise ValueError("b must be non-negative")
    if k < 1:
        raise ValueError("k must be at least 1")

    return pow(a, b, 10 ** k)


print(last_decimal_digits(2, 10, 3))
print(last_decimal_digits(123456789, 987654321, 12))

result = last_decimal_digits(2, 1_000_000_000, 20)
print(f"last 20 digits of 2**1_000_000_000: {result:020d}")


24
132974933589
last 20 digits of 2**1_000_000_000: 91606821041787109376


### Explanation

Since only the remainder modulo `10**k` is needed, three-argument `pow` avoids constructing the enormous full power.


## Problem 14 — Exact perfect-square check

Use `math.isqrt` to test whether large integers are perfect squares without floating-point arithmetic.


In [15]:
# Solution

def is_perfect_square(n: int) -> bool:
    if not isinstance(n, int):
        raise TypeError("n must be an int")
    if n < 0:
        return False
    root = math.isqrt(n)
    return root * root == n


cases = [
    -1,
    0,
    1,
    2,
    16,
    17,
    (10 ** 100 + 12345) ** 2,
    (10 ** 100 + 12345) ** 2 + 1,
]

for n in cases:
    preview = str(n)
    if len(preview) > 50:
        preview = preview[:50] + "..."
    print(f"{preview:>55} -> {is_perfect_square(n)}")


                                                     -1 -> False
                                                      0 -> True
                                                      1 -> True
                                                      2 -> False
                                                     16 -> True
                                                     17 -> False
  10000000000000000000000000000000000000000000000000... -> True
  10000000000000000000000000000000000000000000000000... -> False


### Explanation

`math.isqrt` returns the exact integer floor of the square root and works for arbitrarily large integers. Floating-point square roots can lose precision.


## Problem 15 — Infer internal digit boundaries empirically

Find the first 10 `bit_length` values where `sys.getsizeof(n)` increases. Do not hard-code the answer.


In [16]:
# Solution

def infer_size_boundaries(count: int = 10) -> list[tuple[int, int]]:
    if count < 1:
        raise ValueError("count must be positive")

    boundaries = []
    previous_size = sys.getsizeof(0)
    k = 0

    while len(boundaries) < count:
        n = 2 ** k
        size = sys.getsizeof(n)
        bits = n.bit_length()

        if size != previous_size:
            boundaries.append((bits, size))
            previous_size = size

        k += 1

    return boundaries


boundaries = infer_size_boundaries(10)
gaps = [b2[0] - b1[0] for b1, b2 in zip(boundaries, boundaries[1:])]

print("boundary bit_length | sys.getsizeof")
print("-" * 40)
for bits, size in boundaries:
    print(f"{bits:19d} | {size:13d}")

print()
print("gaps between boundary bit lengths:", gaps)

if gaps:
    likely_digit_bits = statistics.mode(gaps)
    print()
    print(
        "Cautious interpretation: on this Python build, size increases roughly "
        f"every {likely_digit_bits} bits for sufficiently large positive integers."
    )
    print("Treat this as a CPython implementation detail, not portable language behavior.")


boundary bit_length | sys.getsizeof
----------------------------------------
                 31 |            32
                 61 |            36
                 91 |            40
                121 |            44
                151 |            48
                181 |            52
                211 |            56
                241 |            60
                271 |            64
                301 |            68

gaps between boundary bit lengths: [30, 30, 30, 30, 30, 30, 30, 30, 30]

Cautious interpretation: on this Python build, size increases roughly every 30 bits for sufficiently large positive integers.
Treat this as a CPython implementation detail, not portable language behavior.


## Summary of Best Practices

- Use `==` for integer equality, not `is`.
- Use `bit_length()` to reason about integer magnitude.
- Use `bit_count()` instead of converting to binary strings.
- Use `math.isqrt()` for exact integer square-root logic.
- Use three-argument `pow(a, b, m)` for modular exponentiation.
- Validate integer strings before parsing untrusted input.
- Treat exact `sys.getsizeof(int_value)` values as implementation-dependent.
- Benchmark with `timeit.repeat`, multiple repeats, and median summaries.
- Test integer code at boundary values, especially around signed and unsigned limits.
